Model Comparisons for Predicting Campaign Acceptance

In [26]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [27]:
RANDOM_STATE = 1234

In [28]:
PROJECT_ROOT = Path.cwd().parent   # assumes notebook lives in notebooks/, project root is one level up
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_PATH = DATA_DIR / "customer_personality_cleaned.csv"

In [29]:
df = pd.read_csv(RAW_DATA_PATH)

In [30]:
df.columns

Index(['Education', 'Marital_Status', 'Income', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'Complain', 'age', 'Accepted_Cmp', 'duration', 'children',
       'HighestPurchaseSource'],
      dtype='object')

In [31]:
target_col = df['Accepted_Cmp']

In [32]:
feature_cols = df.drop('Accepted_Cmp', axis=1)
categorical_cols = [c for c in feature_cols if df[c].dtype.name in ("category", "object")]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

In [33]:
preprocessor = ColumnTransformer(
    transformers= [
        ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

### Creating a function to initialize each model I plan to use

In [34]:
X = df.drop('Accepted_Cmp', axis=1)
y = (df['Accepted_Cmp'] == "Yes").astype(int)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

results = []

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

#### Evaluate Each Model

In [36]:
# --- KNN ---
knn = KNeighborsClassifier()
pipeline = Pipeline([("preprocess", preprocessor), ("model", knn)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "knn", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"knn: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


knn: mean ROC AUC = 0.7342 (+/- 0.0327)


In [37]:
# --- Logistic Regression ---
log_reg = LogisticRegression(max_iter=2000)
pipeline = Pipeline([("preprocess", preprocessor), ("model", log_reg)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "logistic_regression", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"logistic_regression: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


logistic_regression: mean ROC AUC = 0.7793 (+/- 0.0365)


In [38]:
# --- Logistic Regression (L1) ---
log_reg_l1 = LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000)
pipeline = Pipeline([("preprocess", preprocessor), ("model", log_reg_l1)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "logistic_regression_l1", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"logistic_regression_l1: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


logistic_regression_l1: mean ROC AUC = 0.7797 (+/- 0.0366)


In [39]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", rf)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "random_forest", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"random_forest: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


random_forest: mean ROC AUC = 0.8589 (+/- 0.0245)


In [40]:
# --- Decision Tree ---
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", dt)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "decision_tree", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"decision_tree: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


decision_tree: mean ROC AUC = 0.7100 (+/- 0.0323)


In [41]:
# --- Gradient Boosting ---
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", gb)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "gradient_boosting", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"gradient_boosting: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


gradient_boosting: mean ROC AUC = 0.8429 (+/- 0.0371)


In [42]:
comparison = pd.DataFrame(results).sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
comparison

,model,mean_roc_auc,std_roc_auc
0,random_forest,0.858917,0.024538
1,gradient_boosting,0.842875,0.037117
2,logistic_regression_l1,0.779684,0.036582
3,logistic_regression,0.779341,0.036487
4,knn,0.734170,0.032681
5,decision_tree,0.710016,0.032347


The best model is the Random Forest with a mean roc_auc of 86%. This is what we will use for our prediction

In [43]:
best_model_name = comparison.iloc[0]["model"]
best_model_name

'random_forest'

In [44]:
models = {
    "knn": knn,
    "logistic_regression": log_reg,
    "logistic_regression_l1": log_reg_l1,
    "random_forest": rf,
    "decision_tree": dt,
    "gradient_boosting": gb,
}

best_model = models[best_model_name]
final_pipeline = Pipeline([("preprocess", preprocessor), ("model", best_model)])

final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Income', 'MntWines',
                                                   'MntFruits',
                                                   'MntMeatProducts',
                                                   'MntFishProducts',
                                                   'MntSweetProducts',
                                                   'MntGoldProds',
                                                   'NumDealsPurchases',
                                                   'NumWebPurchases',
                                                   'NumCatalogPurchases',
                                                   'NumStorePurchases',
                                                   'NumWebVisitsMonth',
                                                   'Complain', 'age',
                                                   'duration', 'children']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Education',
                                                   'Marital_Status',
                                                   'HighestPurchaseSource'])])),
                ('model',
                 RandomForestClassifier(n_estimators=300, random_state=1234))])

I have chosen a low threshold to favor recall over precison. The cost of a false positive is just wasted ad spend which is cheaper than a false negative which is not marketing to someone who would have converted. This would be more expensive as it would be a lost sale.

In [45]:
threshold = 0.19

In [46]:
probs = final_pipeline.predict_proba(X_test)[:, 1]
preds = (probs > threshold).astype(int)

print(f"Evaluation of '{best_model_name}' on the held-out test set (threshold = {threshold}):\n")
print(classification_report(y_test, preds, target_names=["No", "Yes"]))
print("Confusion matrix:\n", confusion_matrix(y_test, preds))
print(f"Test ROC AUC: {roc_auc_score(y_test, probs):.4f}")

Evaluation of 'random_forest' on the held-out test set (threshold = 0.19):

              precision    recall  f1-score   support

          No       0.95      0.63      0.76       322
         Yes       0.48      0.91      0.63       121

    accuracy                           0.71       443
   macro avg       0.72      0.77      0.70       443
weighted avg       0.82      0.71      0.72       443

Confusion matrix:
 [[204 118]
 [ 11 110]]
Test ROC AUC: 0.8860


ROC AUC = 0.886 - Strong. This tells us that the model ranks the people likelihood of accepting a campaign. 

For the "Yes": Recall is 0.91 meaning that of everyone who actually accepted the campaign, the model caught 91% of them. However, the precision is 0.48. This means of the 91% flagged as likely to accept a campaign, only 48% actually did.


For the "No": Recall is 0.63 meaning that of everyone who did not accept campaign, the model caught 63% of them. However, the precision is 0.95, which means that of those the model flagged as unlikely to accept a campaign, 95% actually did not accept a campaign.

Since one of the goals of this project is to predict who will accept a campaign, we can say our model did a strong job doing this. The tradeoff, 48% precision, is intentional.

In [47]:
if hasattr(final_pipeline.named_steps["model"], "feature_importances_"):
    feature_names = final_pipeline.named_steps["preprocess"].get_feature_names_out()
    importances = pd.DataFrame({
        "feature": feature_names,
        "importance": final_pipeline.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)
    display(importances.head(15).reset_index(drop=True))
else:
    print(f"{best_model_name} does not expose feature_importances_")

,feature,importance
0,num__MntWines,0.135610
1,num__Income,0.099085
2,num__MntMeatProducts,0.075595
3,num__duration,0.072323
4,num__MntGoldProds,0.071241
5,num__NumCatalogPurchases,0.062978
6,num__MntFishProducts,0.052141
7,num__MntSweetProducts,0.051048
8,num__age,0.050805
9,num__NumStorePurchases,0.050532


The top 4 most Important Features:

Amount of Wine, Income, Amount Meat Products, Duration of Customer Membership 